<a href="https://colab.research.google.com/github/bilgeerol/bim448-sentiment/blob/main/notebooks/02_vader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 – VADER
**Sahibi:** Zehra · **Amaç:** VADER ile İngilizce test setini ve Türkçe test setini (doğrudan ve İngilizceye çeviri sonrası) sınıflandırmak.
**Girdi:** `data/processed/en_test.csv`, `tr_test.csv`, `tr_test_translated.csv`
**Çıktı:** `predictions/vader_en.csv`, `vader_tr_direct.csv`, `vader_tr_mt.csv`, `data/processed/tr_test_translated.csv`

In [ ]:
# 0. KURULUM HÜCRESİ — her oturumda ilk bu çalıştırılır
from google.colab import userdata, drive
import os

drive.mount('/content/drive')

TOKEN = userdata.get('GITHUB_TOKEN')
REPO_OWNER = "bilgeerol"
REPO = "bim448-sentiment"
REPO_DIR = f"/content/{REPO}"

if not os.path.exists(REPO_DIR):
    !git clone https://{TOKEN}@github.com/{REPO_OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git config user.name "zehranurBurmaoglu"
!git config user.email "znburmaoglu@gmail.com"

# Kontrol: depo geldi mi, push iznim var mı?
!ls
!git push --dry-run

Mounted at /content/drive
Cloning into '/content/bim448-sentiment'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 137 (delta 79), reused 90 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 3.91 MiB | 17.72 MiB/s, done.
Resolving deltas: 100% (79/79), done.
/content/bim448-sentiment
Already up to date.
data  notebooks  predictions  README.md  report  requirements.txt  results  src
Everything up-to-date


## 1. VADER İngilizce

In [ ]:
# 1.1 VADER'ı kur, sürümünü kaydet ve birkaç örnekle kuralların etkisini gör
!pip install -q vaderSentiment
import importlib.metadata
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

print("vaderSentiment sürümü:", importlib.metadata.version("vaderSentiment"))
analyzer = SentimentIntensityAnalyzer()

ornekler = [
    "The product is good.",
    "The product is GOOD!!!",
    "The product is not good.",
    "The product is good but very expensive.",
    "I received the product yesterday.",
    "Ürün çok güzel, herkese tavsiye ederim.",
]
for cumle in ornekler:
    print(f"{analyzer.polarity_scores(cumle)}  ←  {cumle}")

vaderSentiment sürümü: 3.3.2
{'neg': 0.0, 'neu': 0.508, 'pos': 0.492, 'compound': 0.4404}  ←  The product is good.
{'neg': 0.0, 'neu': 0.4, 'pos': 0.6, 'compound': 0.6714}  ←  The product is GOOD!!!
{'neg': 0.376, 'neu': 0.624, 'pos': 0.0, 'compound': -0.3412}  ←  The product is not good.
{'neg': 0.0, 'neu': 0.755, 'pos': 0.245, 'compound': 0.2382}  ←  The product is good but very expensive.
{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}  ←  I received the product yesterday.
{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}  ←  Ürün çok güzel, herkese tavsiye ederim.


In [4]:
# 1.2 İngilizce test setini VADER ile sınıflandır ve predictions/vader_en.csv olarak kaydet
import pandas as pd, time

def vader_etiket(compound):
    """VADER'ın standart eşikleri: ≥ 0,05 positive, ≤ −0,05 negative, arası neutral."""
    if compound >= 0.05:
        return "positive"
    if compound <= -0.05:
        return "negative"
    return "neutral"

en_test = pd.read_csv("data/processed/en_test.csv", encoding="utf-8")

bas = time.time()
skorlar = en_test["text"].apply(lambda t: analyzer.polarity_scores(t)["compound"])
sure = time.time() - bas

vader_en = pd.DataFrame({
    "id": en_test["id"],
    "text": en_test["text"],
    "true_label": en_test["label"],
    "pred_label": skorlar.apply(vader_etiket),
    "score": skorlar,
})
vader_en.to_csv("predictions/vader_en.csv", index=False, encoding="utf-8")

# Kontroller: dosyayı diskten tekrar okuyup ortak kurallara uyuyor mu bak
kontrol = pd.read_csv("predictions/vader_en.csv", encoding="utf-8")
print("Satır:", len(kontrol), "| sütunlar:", list(kontrol.columns))
print("id'ler ve sıra test setiyle aynı:", (kontrol["id"].values == en_test["id"].values).all())
print("Geçersiz tahmin etiketi:", (~kontrol["pred_label"].isin(["negative", "neutral", "positive"])).sum())
print(f"Skorlama süresi: {sure:.1f} sn")
print("\nTahmin dağılımı:")
print(kontrol["pred_label"].value_counts().reindex(["negative", "neutral", "positive"]))
print("\ncompound = 0 olan metin sayısı:", (kontrol["score"] == 0).sum())

Satır: 3000 | sütunlar: ['id', 'text', 'true_label', 'pred_label', 'score']
id'ler ve sıra test setiyle aynı: True
Geçersiz tahmin etiketi: 0
Skorlama süresi: 0.7 sn

Tahmin dağılımı:
pred_label
negative     741
neutral      380
positive    1879
Name: count, dtype: int64

compound = 0 olan metin sayısı: 334


In [5]:
# 1.3 VADER İngilizce tahmin dosyasını GitHub'a gönder
!git add predictions/vader_en.csv
!git commit -m "[Zehra] VADER İngilizce tahminleri eklendi"
!git pull --rebase
!git push

[main 9a405b2] [Zehra] VADER İngilizce tahminleri eklendi
 1 file changed, 3001 insertions(+)
 create mode 100644 predictions/vader_en.csv
Current branch main is up to date.
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 221.94 KiB | 4.53 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/bilgeerol/bim448-sentiment.git
   4b15326..9a405b2  main -> main
